# Fase 2 — Fine-tuning DeBERTa-v3-**base** (Kaggle GPU) — run 3

Notebook DELGADO: la lógica vive en `src/` del repo (testeada en local con pytest).
Run 3 tras agotar `small` (~1.07 en 2 runs): **más capacidad** para la señal direccional
de preferencia. **1 época** (ambos runs pusieron el óptimo al final de la época 1),
lr 1e-5, eval cada 25%, ensemble con el baseline al final.

**Setup requerido:**
- Accelerator: GPU T4 x2 o P100 · Internet: ON
- Add-ons > Secrets: `HF_TOKEN` (token de escritura de HF)
- ⚠️ Si ya tenías el notebook en Kaggle: **File > Import Notebook otra vez** — es una copia,
  no se actualiza sola con el repo.

In [ ]:
# Mismas versiones que requirements.txt del repo (reproducibilidad)
!pip -q install "transformers==5.13.0" "datasets==5.0.0" "accelerate==1.14.0" "sentencepiece==0.2.1"
!git clone https://github.com/chamjf234/llm-preference-judge.git repo
import sys; sys.path.insert(0, "repo/src")

# El path del CSV varía entre entornos de Kaggle: detectarlo en vez de asumirlo
from pathlib import Path
CSV = str(next(Path("/kaggle/input").rglob("llm-classification-finetuning/train.csv")))
print("train.csv:", CSV)

In [ ]:
import train
out = train.main(train_csv=CSV, out_dir="/kaggle/working/deberta_out",
                 model_name="microsoft/deberta-v3-base", epochs=1)

In [ ]:
# Ensemble con el baseline (cero GPU): errores decorrelacionados se cancelan al promediar
import ensemble
res = ensemble.blend_with_baseline(CSV, out["val_proba_tta"])

In [ ]:
# Subir pesos a HF Hub — repo NUEVO para base (no pisar los pesos del small)
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForSequenceClassification, AutoTokenizer
token = UserSecretsClient().get_secret("HF_TOKEN")
repo_id = "chamjf234/llm-preference-judge-deberta-v3-base"
AutoModelForSequenceClassification.from_pretrained("/kaggle/working/deberta_out").push_to_hub(repo_id, token=token)
AutoTokenizer.from_pretrained("microsoft/deberta-v3-base").push_to_hub(repo_id, token=token)